In [ ]:
# 必要なものを用意
## ライブラリ
!pip install scikit-learn
## データセット
from datasets import load_dataset
clinc = load_dataset("clinc_oos", "plus")
## 評価関数
from sklearn.metrics import accuracy_score
## パイプライン
from transformers import pipeline
bert_ckpt = "transformersbook/bert-base-uncased-finetuned-clinc"
pipe = pipeline("text-classification", model=bert_ckpt)
## 意図
intents = clinc["test"].features["intent"]
## ベンチマーク用クラス
import torch
from pathlib import Path
from time import perf_counter
import numpy as np
class PerformanceBenchmark:
    def __init__(self, pipeline, dataset, optim_type="BERT baseline"):
        self.pipeline = pipeline
        self.dataset = dataset
        self.optim_type = optim_type

    def compute_accuracy(self):
        preds, labels = [], []
        for example in self.dataset:
            pred = self.pipeline(example["text"])[0]["label"]
            label = example["intent"]
            preds.append(intents.str2int(pred))
            labels.append(label)
        accuracy = accuracy_score(labels, preds)
        print(f"Accuracy on test set - {accuracy:.3f}")
        return {"accuracy": accuracy}

    def compute_size(self):
        temp_path = Path("model.pt")
        torch.save(self.pipeline.model.state_dict(), temp_path)
        size_mb = temp_path.stat().st_size / (1024 * 1024)
        print(f"Model size - {size_mb:.2f} MB")
        temp_path.unlink()
        return {"size_mb": size_mb}

    def time_pipeline(self, query="What is the pin number for my account?"):
        latencies = []
        for _ in range(10):
            _ = self.pipeline(query)
        for _ in range(100):
            start_time = perf_counter()
            _ = self.pipeline(query)
            latency = perf_counter() - start_time
            latencies.append(latency)
        time_avg_ms = 1000 * np.mean(latencies)
        time_std_ms = 1000 * np.std(latencies)
        print(f"Average latency (ms) - {time_avg_ms:.2f} +\- {time_std_ms:.2f}")
        return {"time_avg_ms": time_avg_ms, "time_std_ms": time_std_ms}

    def run_benchmark(self):
        metrics = {}
        metrics[self.optim_type] = self.compute_size()
        metrics[self.optim_type].update(self.time_pipeline())
        metrics[self.optim_type].update(self.compute_accuracy())
        return metrics
## ベースライン
pb = PerformanceBenchmark(pipe, clinc["test"])
perf_metrics = pb.run_benchmark()

In [ ]:
# ハイパーパラメータの追加
from transformers import TrainingArguments

class DistillationTrainingArguments(TrainingArguments):
    def __init__(self, *args, alpha=0.5, temperature=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.alpha = alpha
        self.temperature = temperature

In [ ]:
# カスタムトレーナークラス
import torch.nn as nn
import torch.nn.functional as F
from transformers import Trainer

class DistillationTrainer(Trainer):
    def __init__(self, *args, teacher_model=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher_model = teacher_model
        self.teacher_model.eval()  # 教師モデルは評価モードにする

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        outputs_student = model(**inputs)
        # 生徒からクロスエントロピー損失とロジット抽出
        loss_ce = outputs_student.loss
        logits_student = outputs_student.logits
        # 教師からロジット抽出
        with torch.no_grad():
            outputs_teacher = self.teacher_model(**inputs)
            logits_teacher = outputs_teacher.logits
        # 確率をソフト化し蒸留損失を計算
        loss_function = nn.KLDivLoss(reduction='batchmean') # バッチの次元で損失を平均化
        loss_kd = self.args.temperature ** 2 * loss_function(
            F.log_softmax(logits_student / self.args.temperature, dim=-1),
            F.softmax(logits_teacher / self.args.temperature, dim=-1))
        # 重み付き損失
        loss = self.args.alpha * loss_ce + (1. - self.args.alpha) * loss_kd
        return (loss, outputs_student) if return_outputs else loss

In [ ]:
# クエリの前処理
from transformers import AutoTokenizer

student_ckpt = "distilbert-base-uncased"
student_tokenizer = AutoTokenizer.from_pretrained(student_ckpt)

def tokenize_text(batch):
    return student_tokenizer(batch["text"], truncation=True)

clinc_enc = clinc.map(tokenize_text, batched=True, remove_columns=["text"])
clinc_enc = clinc_enc.rename_column("intent", "labels")

In [ ]:
# Hugging Face ログイン
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
# 評価指標の定義
import numpy as np

def compute_metrics(pred):
    predictions, labels = pred
    predictions = np.argmax(predictions, axis=1)
    return {"accuracy": accuracy_score(labels, predictions)}

In [ ]:
# α = 1 (教師なし)蒸留
batch_size = 48

finetuned_ckpt = "distilbert-base-uncased-finetuned-clinc"
student_training_args = DistillationTrainingArguments(
    output_dir=finetuned_ckpt, eval_strategy = "epoch",
    num_train_epochs=5, learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    alpha=1, weight_decay=0.01,
    push_to_hub=True)

In [ ]:
# 意図とラベルのマッピング
id2label = pipe.model.config.id2label
label2id = pipe.model.config.label2id

In [ ]:
# カスタムモデルの設定
from transformers import AutoConfig

num_labels = intents.num_classes
student_config = (AutoConfig
                  .from_pretrained(student_ckpt, num_labels=num_labels,
                                   id2label=id2label, label2id=label2id))

In [ ]:
# 生徒モデル初期化
from transformers import AutoModelForSequenceClassification
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def student_init():
    return (AutoModelForSequenceClassification
            .from_pretrained(student_ckpt, config=student_config).to(device))

In [ ]:
# ファインチューニング実行
from transformers import DataCollatorWithPadding

teacher_model = (AutoModelForSequenceClassification
                 .from_pretrained(bert_ckpt, num_labels=num_labels)
                 .to(device))

data_collator = DataCollatorWithPadding(tokenizer=student_tokenizer)

distilbert_trainer = DistillationTrainer(model_init=student_init,
    teacher_model=teacher_model, args=student_training_args,
    train_dataset=clinc_enc["train"], eval_dataset=clinc_enc["validation"],
    compute_metrics=compute_metrics,
    data_collator=data_collator, processing_class=student_tokenizer)

distilbert_trainer.train()

In [ ]:
# モデルを Hub にプッシュ
distilbert_trainer.push_to_hub("Training completed!")

In [ ]:
# パイプラインで性能ベンチマーク
finetuned_ckpt = "kirapika2/distilbert-base-uncased-finetuned-clinc"
pipe = pipeline("text-classification", model=finetuned_ckpt)

In [ ]:
# ベンチマークの実行
optim_type = "DistilBERT"
pb = PerformanceBenchmark(pipe, clinc["test"], optim_type=optim_type)
perf_metrics.update(pb.run_benchmark())

In [ ]:
# ベンチマークスコアの比較
import pandas as pd
import matplotlib.pyplot as plt
# プロット関数
def plot_metrics(perf_metrics, current_optim_type):
    df = pd.DataFrame.from_dict(perf_metrics, orient='index')

    for idx in df.index:
        df_opt = df.loc[idx]
        # 現在の最適化タイプを中抜きの円で強調
        if idx == current_optim_type:
            plt.scatter(df_opt["time_avg_ms"], df_opt["accuracy"] * 100,
                        s=df_opt["size_mb"], label=idx,
                        facecolors='none', edgecolors='C0', linewidths=2)
        else:
            plt.scatter(df_opt["time_avg_ms"], df_opt["accuracy"] * 100,
                        s=df_opt["size_mb"], label=idx, alpha=0.5)

    legend = plt.legend(bbox_to_anchor=(1,1))
    for handle in getattr(legend, "legend_handles", getattr(legend, "legendHandles", [])):
        if hasattr(handle, "set_sizes"):
            handle.set_sizes([20])

    plt.ylim(80,90)
    # 最も遅いモデルを使い、x軸のレンジを定義
    xlim = int(perf_metrics["BERT baseline"]["time_avg_ms"] + 3)
    plt.xlim(1,xlim)
    plt.ylabel("Accuracy (%)")
    plt.xlabel("Average latency (ms)")
    plt.show()

plot_metrics(perf_metrics, optim_type)


In [ ]:
# Optuna による Rosenbrock 関数の最適化
!pip install optuna
## objective 関数の定義
def objective(trial):
    x = trial.suggest_float("x", -2, 2)
    y = trial.suggest_float("y", -2, 2)
    return (1 - x) ** 2 + 100 * (y - x ** 2) ** 2

In [ ]:
## study の定義
import optuna

study = optuna.create_study()
study.optimize(objective, n_trials=1000)

In [ ]:
## 最適パラメータ確認
study.best_params

In [ ]:
# ハイパーパラメータ空間の定義
def hp_space(trial):
    return {"num_train_epochs": trial.suggest_int("num_train_epochs", 5, 10),
            "alpha": trial.suggest_float("alpha", 0, 1),
            "temperature": trial.suggest_int("temperature", 2, 20)}

In [ ]:
# ハイパーパラメータの探索
best_run = distilbert_trainer.hyperparameter_search(
    n_trials=20, direction="maximize", hp_space=hp_space)

In [ ]:
# 最適なハイパーパラメータを確認
print(best_run)

In [ ]:
# 最適パラメータでの学習
for k, v in best_run.hyperparameters.items():
    setattr(student_training_args, k, v)

# 蒸留したモデルを格納するリポジトリを新しく定義
distilled_ckpt = "distilbert-base-uncased-distilled-clinc"
student_training_args.output_dir = distilled_ckpt

# 最適なパラメータで新しいトレーナーを作成
distil_trainer = DistillationTrainer(model_init=student_init,
                                     teacher_model=teacher_model, args=student_training_args,
                                     train_dataset=clinc_enc['train'], eval_dataset=clinc_enc['validation'],
                                     compute_metrics=compute_metrics,
                                     data_collator=data_collator, processing_class=student_tokenizer)

# 学習の実行
distil_trainer.train()

In [ ]:
# モデルを Hub にプッシュ
distil_trainer.push_to_hub("Training completed!")

In [ ]:
# 蒸留モデルのベンチマーク
distilled_ckpt = "kirapika2/distilbert-base-uncased-distilled-clinc"
pipe = pipeline("text-classification", model=distilled_ckpt)
optim_type = "Distillation"
pb = PerformanceBenchmark(pipe, clinc["test"], optim_type=optim_type)
perf_metrics.update(pb.run_benchmark())

In [ ]:
# ベンチマーク結果の可視化
plot_metrics(perf_metrics, optim_type)